# Auto-Encoding Variational Bayes (VAE) Implementation
# VAE 구현

**Paper**: Kingma, D.P. & Welling, M. "Auto-Encoding Variational Bayes." arXiv:1312.6114, 2013.

이 노트북은 VAE 논문의 핵심 알고리즘을 PyTorch로 구현합니다. MNIST 데이터셋에 대해 학습하고, latent space 시각화 및 이미지 생성을 수행합니다.

This notebook implements the core VAE algorithm from the paper using PyTorch. We train on MNIST, visualize the latent space, and generate new images.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

## 1. 하이퍼파라미터 설정 / Hyperparameter Setup

논문의 실험 설정을 따릅니다: hidden units 500, minibatch size 100.
Latent dimension은 2D (시각화 용이)와 20D (논문 설정) 두 가지를 테스트합니다.

Following the paper's experimental setup: 500 hidden units, minibatch size 100.
We test both 2D (for visualization) and 20D (paper setting) latent dimensions.

In [ ]:
# Hyperparameters matching the paper.
BATCH_SIZE = 100       # M = 100 in the paper
EPOCHS = 30
LEARNING_RATE = 1e-3
HIDDEN_DIM = 500       # 500 hidden units for MNIST
LATENT_DIM = 2         # Start with 2D for visualization
INPUT_DIM = 784        # 28 x 28 MNIST images
NUM_SAMPLES = 1        # L = 1 in the paper

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. 데이터 로드 / Data Loading

MNIST 데이터셋을 로드합니다. 논문에서는 이진 이미지로 처리하므로, 0-1 범위로 정규화합니다.

Load the MNIST dataset. The paper treats images as binary, so we normalize to [0, 1] range.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} samples, Test: {len(test_dataset)} samples")

## 3. VAE 모델 구현 / VAE Model Implementation

논문 §3과 Appendix C의 구조를 따릅니다:
- **Encoder** (Recognition model, $q_\phi(\mathbf{z}|\mathbf{x})$): MLP → $\boldsymbol{\mu}$, $\log \boldsymbol{\sigma}^2$ 출력 (수식 9, 12)
- **Reparameterization**: $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$ (수식 4, §2.4)
- **Decoder** (Generative model, $p_\theta(\mathbf{x}|\mathbf{z})$): Bernoulli MLP (수식 11)

Following §3 and Appendix C:
- **Encoder** (Recognition model): MLP → outputs $\boldsymbol{\mu}$, $\log \boldsymbol{\sigma}^2$ (eq. 9, 12)
- **Reparameterization**: $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$ (eq. 4, §2.4)
- **Decoder** (Generative model): Bernoulli MLP (eq. 11)

In [ ]:
class VAE(nn.Module):
    """Variational Auto-Encoder following Kingma & Welling (2013).

    Architecture:
        Encoder: x -> h -> (mu, log_var)
        Reparameterize: z = mu + sigma * epsilon
        Decoder: z -> h -> x_recon
    """

    def __init__(
        self,
        input_dim: int = 784,
        hidden_dim: int = 500,
        latent_dim: int = 2,
    ):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: x -> h -> (mu, log_var)
        # Appendix C.2, eq. (12): Gaussian MLP encoder
        self.encoder_hidden = nn.Linear(input_dim, hidden_dim)
        self.encoder_mu = nn.Linear(hidden_dim, latent_dim)
        self.encoder_log_var = nn.Linear(hidden_dim, latent_dim)

        # Decoder: z -> h -> x_recon
        # Appendix C.1, eq. (11): Bernoulli MLP decoder
        self.decoder_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder_output = nn.Linear(hidden_dim, input_dim)

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Encode input to latent distribution parameters.

        Args:
            x: Input tensor of shape (batch_size, input_dim).

        Returns:
            Tuple of (mu, log_var), each of shape (batch_size, latent_dim).
        """
        h = torch.tanh(self.encoder_hidden(x))
        mu = self.encoder_mu(h)
        log_var = self.encoder_log_var(h)
        return mu, log_var

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        """Reparameterization trick (§2.4, eq. 4).

        z = mu + sigma * epsilon, where epsilon ~ N(0, I).
        This allows gradients to flow through the sampling operation.

        Args:
            mu: Mean of the approximate posterior.
            log_var: Log variance of the approximate posterior.

        Returns:
            Sampled latent vector z.
        """
        std = torch.exp(0.5 * log_var)  # sigma = exp(0.5 * log(sigma^2))
        eps = torch.randn_like(std)      # epsilon ~ N(0, I)
        return mu + std * eps             # z = mu + sigma * epsilon

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector to reconstruction.

        Args:
            z: Latent vector of shape (batch_size, latent_dim).

        Returns:
            Reconstructed output of shape (batch_size, input_dim).
        """
        h = torch.tanh(self.decoder_hidden(z))
        return torch.sigmoid(self.decoder_output(h))  # Bernoulli parameter y

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Full forward pass: encode -> reparameterize -> decode.

        Args:
            x: Input tensor of shape (batch_size, input_dim).

        Returns:
            Tuple of (reconstruction, mu, log_var).
        """
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_recon = self.decode(z)
        return x_recon, mu, log_var

## 4. Loss Function: ELBO / 손실 함수: ELBO

논문 수식 (10)의 구현입니다. ELBO = KL regularization + Reconstruction.

Implementation of the paper's equation (10). ELBO = KL regularization + Reconstruction.

$$\mathcal{L}(\theta, \phi; \mathbf{x}^{(i)}) \simeq \frac{1}{2}\sum_{j=1}^{J}\left(1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2\right) + \log p_\theta(\mathbf{x}^{(i)}|\mathbf{z}^{(i)})$$

우리는 이 값을 **최대화**해야 하므로, loss = $-\mathcal{L}$을 **최소화**합니다.

We need to **maximize** this, so we **minimize** loss = $-\mathcal{L}$.

In [ ]:
def vae_loss(
    x_recon: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute VAE loss = -ELBO (eq. 10).

    Args:
        x_recon: Reconstructed input from decoder.
        x: Original input.
        mu: Encoder mean output.
        log_var: Encoder log-variance output.

    Returns:
        Tuple of (total_loss, reconstruction_loss, kl_loss).
    """
    # Reconstruction loss: -E[log p(x|z)]
    # Bernoulli decoder (eq. 11): binary cross-entropy
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction="sum")

    # KL divergence: -(-D_KL) = D_KL(q(z|x) || p(z))
    # Appendix B: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())

    # Total loss = -ELBO = recon_loss + kl_loss
    total_loss = recon_loss + kl_loss

    return total_loss, recon_loss, kl_loss

## 5. 학습 루프 / Training Loop

Algorithm 1 (AEVB)의 구현입니다:
1. $\theta, \phi$ 초기화 / Initialize $\theta, \phi$
2. Minibatch 추출 → noise 샘플링 → gradient 계산 → 파라미터 업데이트
3. 수렴할 때까지 반복 / Repeat until convergence

Implementation of Algorithm 1 (AEVB).

In [ ]:
def train_epoch(
    model: VAE,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> dict[str, float]:
    """Train for one epoch.

    Args:
        model: VAE model.
        loader: Training data loader.
        optimizer: Optimizer.

    Returns:
        Dict with average losses for the epoch.
    """
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    for x, _ in loader:
        x = x.view(-1, INPUT_DIM).to(DEVICE)
        optimizer.zero_grad()

        x_recon, mu, log_var = model(x)
        loss, recon, kl = vae_loss(x_recon, x, mu, log_var)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "recon": total_recon / n,
        "kl": total_kl / n,
    }


def evaluate(
    model: VAE,
    loader: DataLoader,
) -> dict[str, float]:
    """Evaluate model on test set.

    Args:
        model: VAE model.
        loader: Test data loader.

    Returns:
        Dict with average losses.
    """
    model.eval()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    with torch.no_grad():
        for x, _ in loader:
            x = x.view(-1, INPUT_DIM).to(DEVICE)
            x_recon, mu, log_var = model(x)
            loss, recon, kl = vae_loss(x_recon, x, mu, log_var)
            total_loss += loss.item()
            total_recon += recon.item()
            total_kl += kl.item()

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "recon": total_recon / n,
        "kl": total_kl / n,
    }

In [ ]:
# Initialize model and optimizer.
model = VAE(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Latent dim: {LATENT_DIM}, Hidden dim: {HIDDEN_DIM}")

In [ ]:
# Training loop.
train_history = []
test_history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_epoch(model, train_loader, optimizer)
    test_metrics = evaluate(model, test_loader)

    train_history.append(train_metrics)
    test_history.append(test_metrics)

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:3d} | "
            f"Train ELBO: {-train_metrics['loss']:.1f} | "
            f"Test ELBO: {-test_metrics['loss']:.1f} | "
            f"Recon: {test_metrics['recon']:.1f} | "
            f"KL: {test_metrics['kl']:.1f}"
        )

## 6. 학습 곡선 시각화 / Training Curves

논문 Figure 2와 유사하게, variational lower bound (ELBO)의 수렴 과정을 시각화합니다.
또한 KL divergence와 reconstruction loss의 변화도 함께 보여줍니다.

Similar to Figure 2 in the paper, we visualize the convergence of the ELBO.
We also show the evolution of KL divergence and reconstruction loss.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = range(1, EPOCHS + 1)

# ELBO (negated loss)
axes[0].plot(epochs_range, [-t["loss"] for t in train_history], label="Train")
axes[0].plot(epochs_range, [-t["loss"] for t in test_history], label="Test")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("ELBO")
axes[0].set_title("Variational Lower Bound")
axes[0].legend()

# Reconstruction loss
axes[1].plot(epochs_range, [t["recon"] for t in train_history], label="Train")
axes[1].plot(epochs_range, [t["recon"] for t in test_history], label="Test")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Reconstruction Loss")
axes[1].set_title("Reconstruction (BCE)")
axes[1].legend()

# KL divergence
axes[2].plot(epochs_range, [t["kl"] for t in train_history], label="Train")
axes[2].plot(epochs_range, [t["kl"] for t in test_history], label="Test")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("KL Divergence")
axes[2].set_title("KL(q(z|x) || p(z))")
axes[2].legend()

plt.tight_layout()
plt.show()

## 7. 재구성 결과 시각화 / Reconstruction Visualization

원본 이미지와 VAE 재구성 결과를 비교합니다.
Encoder → reparameterization → decoder의 전체 파이프라인 결과입니다.

Compare original images with VAE reconstructions.
This shows the full encoder → reparameterization → decoder pipeline.

In [ ]:
model.eval()
with torch.no_grad():
    x_test, _ = next(iter(test_loader))
    x_test = x_test.view(-1, INPUT_DIM).to(DEVICE)
    x_recon, _, _ = model(x_test)

n_show = 10
fig, axes = plt.subplots(2, n_show, figsize=(15, 3))

for i in range(n_show):
    # Original
    axes[0, i].imshow(x_test[i].cpu().view(28, 28), cmap="gray")
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("Original", fontsize=10)

    # Reconstruction
    axes[1, i].imshow(x_recon[i].cpu().view(28, 28), cmap="gray")
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("Reconstructed", fontsize=10)

plt.suptitle("Original vs Reconstructed (Top: Original, Bottom: VAE)")
plt.tight_layout()
plt.show()

## 8. Latent Space 시각화 / Latent Space Visualization

### 8-1. 2D Latent Space Encoding

$J = 2$인 경우, 테스트 데이터의 latent representation을 2D 평면에 시각화합니다.
각 숫자 클래스가 latent space에서 어떻게 분포하는지 확인합니다.

For $J = 2$, we visualize the latent representations of test data on a 2D plane.
We observe how each digit class is distributed in latent space.

In [ ]:
if LATENT_DIM == 2:
    model.eval()
    z_list = []
    label_list = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.view(-1, INPUT_DIM).to(DEVICE)
            mu, _ = model.encode(x)
            z_list.append(mu.cpu())
            label_list.append(y)

    z_all = torch.cat(z_list).numpy()
    labels_all = torch.cat(label_list).numpy()

    plt.figure(figsize=(8, 8))
    scatter = plt.scatter(
        z_all[:, 0], z_all[:, 1],
        c=labels_all, cmap="tab10", s=5, alpha=0.6
    )
    plt.colorbar(scatter, label="Digit")
    plt.xlabel("$z_1$")
    plt.ylabel("$z_2$")
    plt.title("Latent Space: Test Data Encoded by Digit Class")
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print(f"Latent dim = {LATENT_DIM}, skipping 2D scatter plot.")

### 8-2. Learned Manifold 시각화 / Learned Manifold Visualization

논문 Figure 4(b)를 재현합니다. 2D latent space에서 균등하게 배치한 좌표를
Gaussian inverse CDF로 변환하고, decoder를 통해 이미지를 생성합니다.

Reproducing Figure 4(b) from the paper. We place uniformly spaced coordinates
in 2D latent space, transform through Gaussian inverse CDF, and generate
images via the decoder.

In [ ]:
if LATENT_DIM == 2:
    n_grid = 20
    # Uniformly spaced percentiles, transformed via inverse CDF (ppf).
    # This matches the paper's approach in Appendix A / Figure 4.
    grid = norm.ppf(np.linspace(0.05, 0.95, n_grid))

    canvas = np.zeros((28 * n_grid, 28 * n_grid))

    model.eval()
    with torch.no_grad():
        for i, z1 in enumerate(grid):
            for j, z2 in enumerate(grid):
                z = torch.tensor([[z1, z2]], dtype=torch.float32).to(DEVICE)
                x_decoded = model.decode(z)
                digit = x_decoded.cpu().view(28, 28).numpy()
                canvas[
                    (n_grid - 1 - i) * 28 : (n_grid - i) * 28,
                    j * 28 : (j + 1) * 28,
                ] = digit

    plt.figure(figsize=(10, 10))
    plt.imshow(canvas, cmap="gray")
    plt.title("Learned MNIST Manifold (2D Latent Space)")
    plt.xlabel("$z_1$")
    plt.ylabel("$z_2$")
    plt.xticks(
        np.arange(0, 28 * n_grid, 28 * 4),
        [f"{g:.1f}" for g in grid[::4]],
    )
    plt.yticks(
        np.arange(0, 28 * n_grid, 28 * 4),
        [f"{g:.1f}" for g in grid[::-4]],
    )
    plt.show()
else:
    print(f"Latent dim = {LATENT_DIM}, skipping manifold visualization.")

## 9. 새로운 이미지 생성 / Generating New Images

Prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$에서 $\mathbf{z}$를 샘플링하고
decoder를 통해 새로운 이미지를 생성합니다. 논문 Figure 5와 유사합니다.

Sample $\mathbf{z}$ from the prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$
and generate new images through the decoder. Similar to Figure 5 in the paper.

In [ ]:
model.eval()
n_samples = 20

with torch.no_grad():
    # Sample from prior p(z) = N(0, I)
    z_samples = torch.randn(n_samples, LATENT_DIM).to(DEVICE)
    generated = model.decode(z_samples)

fig, axes = plt.subplots(2, n_samples // 2, figsize=(15, 3))
for i in range(n_samples):
    row = i // (n_samples // 2)
    col = i % (n_samples // 2)
    axes[row, col].imshow(generated[i].cpu().view(28, 28), cmap="gray")
    axes[row, col].axis("off")

plt.suptitle(f"Generated Samples from Prior (Latent dim = {LATENT_DIM})")
plt.tight_layout()
plt.show()

## 10. Reparameterization Trick 검증 / Verifying the Reparameterization Trick

Reparameterization trick의 핵심은 gradient가 $\phi$ (encoder 파라미터)로 흐를 수 있다는 것입니다.
아래에서 이를 실제로 확인합니다.

The key property of the reparameterization trick is that gradients can flow to $\phi$ (encoder parameters).
We verify this below.

In [ ]:
# Demonstrate that gradients flow through the reparameterization trick.
x_sample = torch.randn(1, INPUT_DIM, requires_grad=False).to(DEVICE)

mu, log_var = model.encode(x_sample)
z = model.reparameterize(mu, log_var)

# Compute a scalar loss from z.
dummy_loss = z.sum()
dummy_loss.backward()

# Check that encoder parameters have gradients.
has_grad = all(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in [model.encoder_hidden.weight, model.encoder_mu.weight, model.encoder_log_var.weight]
)
print(f"Gradients flow through reparameterization: {has_grad}")

# Show gradient magnitudes.
for name, param in [("encoder_hidden", model.encoder_hidden.weight),
                     ("encoder_mu", model.encoder_mu.weight),
                     ("encoder_log_var", model.encoder_log_var.weight)]:
    print(f"  {name}: grad norm = {param.grad.norm():.6f}")

model.zero_grad()

## 11. KL Divergence 해석적 계산 검증 / Verifying Analytical KL Divergence

Appendix B의 해석적 KL 공식이 Monte Carlo 추정과 일치하는지 확인합니다.

Verify that the analytical KL formula from Appendix B matches Monte Carlo estimation.

In [ ]:
# Analytical KL vs Monte Carlo KL comparison.
mu_test = torch.tensor([[0.5, -0.3]])
log_var_test = torch.tensor([[-1.0, -0.5]])

# Analytical (Appendix B):
# D_KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
kl_analytical = -0.5 * torch.sum(
    1 + log_var_test - mu_test.pow(2) - log_var_test.exp()
).item()

# Monte Carlo estimation with many samples.
n_mc = 100_000
std_test = torch.exp(0.5 * log_var_test)
z_mc = mu_test + std_test * torch.randn(n_mc, 2)

# log q(z|x) - log p(z)
log_q = -0.5 * torch.sum(
    log_var_test + (z_mc - mu_test).pow(2) / log_var_test.exp(), dim=1
) - 0.5 * 2 * np.log(2 * np.pi)
log_p = -0.5 * torch.sum(z_mc.pow(2), dim=1) - 0.5 * 2 * np.log(2 * np.pi)
kl_mc = (log_q - log_p).mean().item()

print(f"KL (analytical, Appendix B): {kl_analytical:.4f}")
print(f"KL (Monte Carlo, {n_mc:,} samples): {kl_mc:.4f}")
print(f"Difference: {abs(kl_analytical - kl_mc):.4f}")

## 12. Latent Dimension 비교 실험 / Latent Dimension Comparison

논문 Figure 2처럼 다양한 latent dimension ($N_z = 2, 5, 10, 20$)에서의 ELBO를 비교합니다.
Latent dimension이 증가해도 overfitting이 없는 것을 확인합니다 (KL regularization 효과).

Like Figure 2, we compare ELBO across different latent dimensions ($N_z = 2, 5, 10, 20$).
We verify that increasing latent dimensions does not cause overfitting (KL regularization effect).

In [ ]:
latent_dims = [2, 5, 10, 20]
comparison_epochs = 20
comparison_results = {}

for ldim in latent_dims:
    print(f"\nTraining with latent_dim = {ldim}...")
    m = VAE(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=ldim).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LEARNING_RATE)

    history = []
    for epoch in range(1, comparison_epochs + 1):
        train_epoch(m, train_loader, opt)
        test_met = evaluate(m, test_loader)
        history.append(test_met)
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}: Test ELBO = {-test_met['loss']:.1f}")

    comparison_results[ldim] = history

# Plot comparison.
plt.figure(figsize=(8, 5))
for ldim, hist in comparison_results.items():
    elbos = [-h["loss"] for h in hist]
    plt.plot(range(1, comparison_epochs + 1), elbos, label=f"$N_z = {ldim}$")

plt.xlabel("Epoch")
plt.ylabel("Test ELBO")
plt.title("ELBO vs Latent Dimension (cf. Figure 2)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 13. 요약 / Summary

### 구현한 내용 / What We Implemented

1. **VAE 모델** (§3): Gaussian encoder + Bernoulli decoder MLP
2. **Reparameterization trick** (§2.4): $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$
3. **ELBO loss** (수식 10): analytical KL (Appendix B) + BCE reconstruction
4. **AEVB training** (Algorithm 1): minibatch SGD with Adam optimizer

### 확인한 결과 / What We Verified

1. Reparameterization trick을 통한 gradient 전파 / Gradient flow through reparameterization
2. 해석적 KL과 Monte Carlo KL의 일치 / Analytical vs Monte Carlo KL agreement
3. Latent dimension 증가 시 overfitting 없음 / No overfitting with increased latent dimensions
4. 2D latent space의 smooth manifold / Smooth manifold in 2D latent space
5. Prior에서 sampling한 새로운 이미지 생성 / New image generation from prior samples

### 논문과의 차이점 / Differences from the Paper

| 항목 / Item | 논문 / Paper | 구현 / Implementation |
|---|---|---|
| Optimizer | Adagrad | Adam (더 일반적으로 사용됨) |
| Activation | tanh (encoder & decoder) | tanh (논문과 동일) |
| Initialization | $\mathcal{N}(0, 0.01)$ | PyTorch 기본값 |
| Weight decay | $p(\theta) = \mathcal{N}(0, I)$ | 없음 (간결성 위해) |